In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================
# Config
# =========================

@dataclass(frozen=True)
class Paths:
    main_workbook: Path = Path("data") / "data.xlsx"
    output_dir: Path = Path("outputs") / "figure_5"
    positive_sheet: str = "Positive"


CLASS_ORDER = ["Lamp oil", "White spirit", "Brandspiritus", "Gasoline", "Diesel"]

SPECTRAL_AXIS_LABEL = "Wavelength (nm)"
X_MIN, X_MAX = 1300.0, 2600.0


# =========================
# Root sets for class mapping (Positive sheet)
# =========================

DIESEL_ROOTS = {
    "SH3", "SH4", "SH7", "SH8", "SH11", "SH12", "SH15", "SH16", "SH19", "SH20",
    "T3", "T6", "T9", "T12", "T15",
    "Te3", "Te6", "Te9", "Te12", "Te15",
}

GASOLINE_95_ROOTS = {
    "SH1", "SH5", "SH9", "SH13", "SH17",
    "T1", "T4", "T7", "T10", "T13",
    "Te1", "Te4", "Te7", "Te10", "Te13",
}

GASOLINE_98_ROOTS = {
    "SH2", "SH6", "SH10", "SH14", "SH18",
    "T2", "T5", "T8", "T11", "T14",
    "Te2", "Te5", "Te8", "Te11", "Te14",
}

GASOLINE_ROOTS = GASOLINE_95_ROOTS.union(GASOLINE_98_ROOTS)


# =========================
# Helpers
# =========================

def root_code(sample_id: str) -> str:
    return str(sample_id).strip().split("-", 1)[0]


def class_label_from_root(root: str) -> str:
    r = str(root)
    if r.startswith("B"):
        return "Brandspiritus"
    if r in DIESEL_ROOTS:
        return "Diesel"
    if r in GASOLINE_ROOTS:
        return "Gasoline"
    if r.startswith("L"):
        return "Lamp oil"
    if r.startswith("W"):
        return "White spirit"
    raise ValueError(f"Unknown root code for class mapping: {root}")


def get_sorted_numeric_spectral_columns(df: pd.DataFrame) -> Tuple[List[str], np.ndarray]:
    numeric_cols: List[str] = []
    numeric_axis: List[float] = []
    for col in df.columns:
        try:
            x = float(col)
        except (TypeError, ValueError):
            continue
        numeric_cols.append(col)
        numeric_axis.append(x)

    if not numeric_cols:
        raise ValueError("No numeric spectral columns found (headers must be numeric strings).")

    axis_arr = np.array(numeric_axis, dtype=float)
    sort_idx = np.argsort(axis_arr)
    sorted_cols = [numeric_cols[i] for i in sort_idx]
    sorted_axis = axis_arr[sort_idx]
    return sorted_cols, sorted_axis


def load_all_positive_samples(paths: Paths) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Load all samples from the Positive worksheet."""
    df = pd.read_excel(paths.main_workbook, sheet_name=paths.positive_sheet, index_col=0)
    df.index = df.index.to_series().astype(str)

    roots = df.index.to_series().map(root_code).astype(str)

    sorted_cols, x_axis = get_sorted_numeric_spectral_columns(df)
    X_raw = df[sorted_cols].to_numpy(dtype=float)
    y = np.array([class_label_from_root(r) for r in roots.to_numpy()], dtype=str)
    return X_raw, y, x_axis


# =========================
# Plotting: raw mean spectra overlay
# =========================

def plot_raw_mean_spectra_overlay(
    x_axis: np.ndarray,
    X_raw: np.ndarray,
    y: np.ndarray,
    out_dir: Path,
) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(8.2, 4.8))
    cmap = plt.get_cmap("tab10")

    for i, cls in enumerate(CLASS_ORDER):
        mask = (y == cls)
        if not np.any(mask):
            continue
        mean_spec = np.mean(X_raw[mask, :], axis=0)
        ax.plot(x_axis, mean_spec, linewidth=1.6, color=cmap(i), label=cls)

    ax.set_xlabel(SPECTRAL_AXIS_LABEL)
    ax.set_ylabel("Absorbance (a.u.)")
    ax.set_xlim(X_MIN, X_MAX)
    legend = ax.legend(loc="best", frameon=True)
    legend.get_frame().set_linewidth(0.8)
    legend.get_frame().set_edgecolor("black")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", which="both", direction="in")
    fig.tight_layout()

    out_path = out_dir / "mean_spectra_overlay_raw_all_samples.png"
    fig.savefig(out_path, dpi=300)
    plt.close(fig)
    return out_path


def run(paths: Paths) -> None:
    X_raw, y, x_axis = load_all_positive_samples(paths)
    out_path = plot_raw_mean_spectra_overlay(
        x_axis=x_axis,
        X_raw=X_raw,
        y=y,
        out_dir=paths.output_dir,
    )
    print(f"[OK] Saved figure to: {out_path}")


if __name__ == "__main__":
    cfg = Paths()
    run(cfg)
